In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# Import for class weight calculation and signal analysis
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import os

# --- Configuration ---
# NOTE: The script assumes FEATURE_FILE and TRUESKILL_FILE exist in the current directory
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "ai_rank_gpt.csv"
RANDOM_STATE = 42
TOTAL_CLASSES = 3

print(f"--- Starting Robust NN Training using {FEATURE_FILE} ---")

# 1. Load Data
try:
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name').fillna(0.0)
    ts_df = pd.read_csv(TRUESKILL_FILE).set_index('image_name')[['ai_ts_score']].rename(columns={'ai_ts_score': 'safety_score'})
except FileNotFoundError as e:
    print(f"\nFATAL ERROR: Required file not found. Ensure '{FEATURE_FILE}' and '{TRUESKILL_FILE}' exist and are loaded.")
    raise e

print(f"Feature DataFrame loaded (Shape: {X_df.shape}). NaN values successfully filled with 0.0.")

# 2. Target Variable (Safety Bin) Preparation
# Re-calculate the 10 safety bins to ensure consistency
minv = ts_df['safety_score'].min()
maxv = ts_df['safety_score'].max()
bins = np.linspace(minv, maxv, TOTAL_CLASSES + 1)   # 11 edges -> 10 equal-width bins

ts_df['safety_bin_1_10'] = pd.cut(
    ts_df['safety_score'],
    bins=bins,
    labels=range(1, TOTAL_CLASSES + 1),    # labels 1..10
    include_lowest=True
).astype(int)
counts = ts_df['safety_bin_1_10'].value_counts(sort=False).reindex(range(1,TOTAL_CLASSES + 1), fill_value=0)
print("Counts per equal-width safety bin (1..10):")
print(counts)

# 3. print bin ranges so you know what each label means
print("\nLabel -> numeric range (inclusive left, exclusive right except last):")
for i in range(TOTAL_CLASSES):
    left, right = bins[i], bins[i+1]
    # close the last interval on the right for readability
    right_bracket = ']' if i == 9 else ')'
    print(f"Label {i+1}: [{left:.6g}, {right:.6g}{right_bracket}")

# 4. bar plot (histogram of counts per label)
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Safety bin (equal-width, 1 = lowest range)')
ax.set_ylabel('Number of features')
ax.set_title('Number of features in each equal-width safety bin (1..10)')
ax.set_xticklabels([str(i) for i in range(1,TOTAL_CLASSES + 1)], rotation=0)
plt.tight_layout()
plt.show()

# min_score = ts_df['safety_score'].min()
# max_score = ts_df['safety_score'].max()
# bins = np.linspace(min_score, max_score, TOTAL_CLASSES + 1)
# ts_df['safety_bin_1_10'] = pd.cut(
#     ts_df['safety_score'],
#     bins=bins,
#     labels=False,
#     include_lowest=True
# )
# The y_raw_full series contains the 0-9 labels indexed by image_name
y_raw_full = ts_df['safety_bin_1_10']

# 3. Final Merge and Alignment
combined_df = X_df.merge(y_raw_full, left_index=True, right_index=True, how='inner')

# Find the target column name
target_col_name = [col for col in combined_df.columns if 'safety_bin_1_10' in col][-1]

X = combined_df.drop(columns=[target_col_name]).values
y_raw = combined_df[target_col_name].values
y_raw = y_raw.astype(int) - 1
# 4. Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\n[4/8] Features scaled using StandardScaler.")

# --- Data Split: 80% Train, 10% Validation, 10% Test ---

# 5. Split 1: 80% Train, 20% Temp (Test + Validation)
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_raw
)

# 6. Split 2: Divide Temp (20%) into Test (10%) and Validation (10%) (0.5 means 50% of 20% = 10% total)
X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_raw
)

# 7. One-Hot Encoding the Target Sets
y_train = to_categorical(y_train_raw, num_classes=TOTAL_CLASSES)
y_val = to_categorical(y_val_raw, num_classes=TOTAL_CLASSES)
y_test = to_categorical(y_test_raw, num_classes=TOTAL_CLASSES)
print("[7/8] Target variables One-Hot Encoded.")


# 8. Calculate Class Weights (Crucial for Imbalanced Data)
classes = np.unique(y_train_raw)
class_weights_array = compute_class_weight(
    'balanced',
    classes=classes,
    y=y_train_raw
)
class_weight_dict = dict(zip(classes, class_weights_array))
print("\n[8/8] Class weights calculated to handle data imbalance:")
print(class_weight_dict)

# --- Summary of Prepared Data ---
print("\n--- Summary of Prepared Data for Robust Training ---")
print(f"Training Features (X_train) Shape: {X_train.shape} (80%)")
print(f"Validation Features (X_val) Shape: {X_val.shape} (10%)")
print(f"Testing Features (X_test) Shape: {X_test.shape} (10%)")


# --- Robust Model Definition ---
def create_robust_model(input_dim, output_dim):
    """Creates a deeper, more robust feed-forward neural network."""
    model = Sequential([
        # Input Layer: Wider layer to capture initial feature interactions
        Dense(256, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        # Hidden Layer 1: Increased depth
        Dense(128, activation='relu'),
        Dropout(0.3),
        # Hidden Layer 2
        Dense(64, activation='relu'),
        # Output layer
        Dense(output_dim, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create and summarize the model
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]
model = create_robust_model(input_dim, output_dim)

print("\n--- Robust Neural Network Model Summary ---")
model.summary()

# --- Callbacks for Robust Training ---
# Early Stopping: Stop training when validation loss stops improving (patience=10 epochs)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

# ReduceLROnPlateau: Reduce learning rate if val_loss plateaus
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

callbacks = [early_stop, reduce_lr]

# --- Training the Robust Model ---
print("\n--- Starting Robust Model Training (Max 100 Epochs) ---")
history = model.fit(
    X_train, y_train,
    epochs=100, # Max epochs set high, relying on EarlyStopping
    batch_size=32,
    validation_data=(X_val, y_val), # Use the dedicated validation set
    class_weight=class_weight_dict, # Apply calculated weights for imbalance
    callbacks=callbacks,
    verbose=1
)

# --- Final Evaluation on the Test Set ---
print("\n--- Final Evaluation on Dedicated 10% Test Set ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# --- Comprehensive Classification Report ---
print("\n--- Comprehensive Classification Report (Test Set) ---")
y_pred_probs = model.predict(X_test)
y_test_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred_probs, axis=1)

target_names = [f'Bin {i}' for i in range(1, TOTAL_CLASSES + 1)]
# Set zero_division=0 to handle cases where a bin has no predictions/true samples
print(classification_report(y_test_labels, y_pred_labels, target_names=target_names, labels=range(TOTAL_CLASSES), zero_division=0))

# --- PLOTTING THE CONFUSION MATRIX  ---
print("\n--- Plotting Confusion Matrix ---")

# Calculate the confusion matrix
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=range(TOTAL_CLASSES))

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(TOTAL_CLASSES, TOTAL_CLASSES))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix (Test Set)', fontsize=16)
plt.show()

# --- PLOTTING THE TRAINING CURVES  ---
print("\n--- Generating Training and Validation Curves ---")

# Extract metrics from the history object
train_acc = history.history['accuracy']
train_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
epochs = range(1, len(train_acc) + 1)

# Find the best epoch based on minimum validation loss (for plotting marker)
best_epoch_index = np.argmin(val_loss)
best_epoch = best_epoch_index + 1

plt.figure(figsize=(14, 6))

# Subplot 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs, train_acc, 'go-', label='Training Accuracy', linewidth=2, alpha=0.8)
plt.plot(epochs, val_acc, 'bo-', label='Validation Accuracy', linewidth=2, alpha=0.8)
plt.plot(best_epoch, val_acc[best_epoch_index], 'r*', markersize=15, label=f'Best Epoch ({best_epoch})')

plt.title('Training and Validation Accuracy', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)

# Subplot 2: Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, train_loss, 'r^-', label='Training Loss', linewidth=2, alpha=0.8)
plt.plot(epochs, val_loss, 'c^-', label='Validation Loss', linewidth=2, alpha=0.8)
plt.plot(best_epoch, val_loss[best_epoch_index], 'm*', markersize=15, label=f'Best Epoch ({best_epoch})')

plt.title('Training and Validation Loss', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (Categorical Crossentropy)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

model.save("my_model.keras")

print("\n--- Robust Training and Evaluation Complete ---")
print("Check the Test Accuracy and the Classification Report. The class weights should now help improve performance in the previously ignored bins.")

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model

from IPython.display import HTML

# --- Configuration ---
FEATURE_FILE = "processed_features_conf_0.7.csv"
TRUESKILL_FILE = "ai_rank_gpt.csv"
MODEL_FILE = "my_model.keras"
# Base URL for public image access
BASE_IMAGE_URL = "https://cycleai.net/london/training/training/Images/"
TOTAL_CLASSES = 3

print("--- Starting Inference Pipeline ---")

# 1. Load Data
try:
    X_df = pd.read_csv(FEATURE_FILE, index_col='image_name').fillna(0.0)
    ts_df = pd.read_csv(TRUESKILL_FILE).set_index('image_name')[['ai_ts_score']].rename(columns={'ai_ts_score': 'safety_score'})
except FileNotFoundError:
    print("Error: Input CSV files not found.")
    exit()

print(f"Diagnostic: Feature file (X_df) loaded with {len(X_df)} images.")
print(f"Diagnostic: TrueSkill file (ts_df) loaded with {len(ts_df)} scores.")

# 2. Replicate Training Processing (Crucial for consistency)
# We must perform the exact same binning to know what "Bin 0", "Bin 1", etc. represent.
try:
    minv = ts_df['safety_score'].min()
    maxv = ts_df['safety_score'].max()

    # 1. Re-calculate equal-width bin edges based on TOTAL_CLASSES
    # (TOTAL_CLASSES + 1) edges for TOTAL_CLASSES bins
    bin_edges = np.linspace(minv, maxv, TOTAL_CLASSES + 1)

    # 2. Use pd.cut to assign the labels to a NEW column named 'safety_bin'
    # We store the *labels* in the DataFrame, and the *edges* in bin_edges (defined above)
    ts_df['safety_bin'] = pd.cut(
        ts_df['safety_score'],
        bins=bin_edges,  # Use the calculated edges
        labels=range(1, TOTAL_CLASSES + 1), # labels 1, 2, 3
        include_lowest=True
    ).astype(int)

    # 3. Print counts for verification (optional, but good practice)
    counts = ts_df['safety_bin'].value_counts(sort=False).reindex(range(1, TOTAL_CLASSES + 1), fill_value=0)
    print(f"Recovered {len(bin_edges)-1} bins from training logic.")
    print(f"Bin edges: {np.round(bin_edges, 3)}")

except ValueError as e:
    print(f"Error reproducing binning logic: {e}")
    exit()

# 3. Fit Scaler on Training Data
# The scaler must be fit on the exact same data used to train the model.
# The inner merge finds the set of images that have both features AND valid labels.
combined_df = X_df.merge(ts_df[['safety_bin']], left_index=True, right_index=True, how='inner')

# Training features (X) used for fitting the scaler
X_train_raw = combined_df.drop(columns=['safety_bin']).values

scaler = StandardScaler()
scaler.fit(X_train_raw)
print(f"Scaler re-fitted on {len(X_train_raw)} labeled images (combined_df size).")

# 4. Select 10 Images for Prediction
# Unseen = Images in original Feature File (X_df) that were NOT part of the final training set (combined_df)
unseen_indices = X_df.index.difference(combined_df.index)

print(f"Diagnostic: Found {len(unseen_indices)} images that are in the feature file but not the training set.")

if len(unseen_indices) == 0:
    print("\nWARNING: No new/unseen images found. This means all images in your feature file also have a score.")
    print("Falling back to selecting 10 random images from the full feature set (X_df) to demonstrate the pipeline.")
    sample_indices = np.random.choice(X_df.index, size=10, replace=False)
elif len(unseen_indices) < 10:
    print(f"Not enough unseen images found (only {len(unseen_indices)}). Prediction will be run on these.")
    sample_indices = unseen_indices
else:
    # Randomly sample 10 unseen images
    sample_indices = np.random.choice(unseen_indices, size=10, replace=False)

print(f"\nSelected 10 images for prediction: {sample_indices}")

# 5. Prepare Features for the Sample
X_sample_raw = X_df.loc[sample_indices].values
X_sample_scaled = scaler.transform(X_sample_raw)

# 6. Load Model and Predict
try:
    if not os.path.exists(MODEL_FILE):
        print(f"Model file '{MODEL_FILE}' not found. Please ensure the training script ran successfully.")
        exit()

    model = load_model(MODEL_FILE)
    predictions_probs = model.predict(X_sample_scaled, verbose=0)

    # Get the predicted bin index
    predicted_bins = np.argmax(predictions_probs, axis=1)

    # Calculate Estimated Score based on qcut edges
    # We take the midpoint of the predicted bin's edges
    estimated_scores = []
    for bin_idx in predicted_bins:
        # Safety check: ensure predicted bin is within the edges we recovered
        if bin_idx < len(bin_edges) - 1:
            lower = bin_edges[bin_idx]
            upper = bin_edges[bin_idx + 1]
            midpoint = (lower + upper) / 2
            estimated_scores.append(midpoint)
        else:
            # Fallback for out-of-bounds prediction (e.g., if model predicts bin 9 but only 6 bins were created)
            print(f"Warning: Predicted bin {bin_idx} is outside the known range {len(bin_edges)-1}. Using max edge.")
            estimated_scores.append(bin_edges[-1])

except Exception as e:
    print(f"Error during inference: {e}")
    exit()

# 7. Display Results with Image URL in a formatted HTML table
print("\n--- Prediction Results (Visual Display) ---")

html_content = """
<style>
    .prediction-table { border-collapse: collapse; width: 100%; font-family: sans-serif; }
    .prediction-table th, .prediction-table td { border: 1px solid #ddd; padding: 8px; text-align: left; }
    .prediction-table th { background-color: #f2f2f2; color: #333; }
    .prediction-image { max-width: 150px; height: auto; border-radius: 4px; }
    .score-container { display: flex; align-items: center; }
</style>
<table class="prediction-table">
    <thead>
        <tr>
            <th>Image Name</th>
            <th>Predicted Bin</th>
            <th>Est. Safety Score</th>
            <th>Image Preview</th>
        </tr>
    </thead>
    <tbody>
"""

for i, img_name in enumerate(sample_indices):
    bin_idx = predicted_bins[i]
    score = estimated_scores[i]
    image_url = f"{BASE_IMAGE_URL}{img_name}"

    # Calculate color for visual representation (normalized 0 to 100, then mapped to a green/red gradient)
    # This is a heuristic mapping for visualization
    # We use a simple linear interpolation based on the score bounds recovered from qcut
    min_score = bin_edges[0]
    max_score = bin_edges[-1]

    # Normalize score between 0 and 1
    if max_score > min_score:
        normalized_score = (score - min_score) / (max_score - min_score)
    else:
        normalized_score = 0.5

    # Interpolate from Red (0, unsafe) to Green (1, safe)
    red = int(255 * (1 - normalized_score))
    green = int(255 * normalized_score)
    score_color = f"rgb({red}, {green}, 0)"

    html_content += f"""
        <tr>
            <td>{img_name}</td>
            <td>Bin {bin_idx}</td>
            <td>
                <div class="score-container">
                    <span style="background-color: {score_color}; color: white; padding: 3px 6px; border-radius: 4px; margin-right: 10px; font-weight: bold;">
                        {score:.4f}
                    </span>
                    <small>(Range: {bin_edges[bin_idx]:.2f} - {bin_edges[bin_idx+1]:.2f})</small>
                </div>
            </td>
            <td>
                <a href="{image_url}" target="_blank">
                    <img src="{image_url}" class="prediction-image" alt="Preview of {img_name}">
                </a>
            </td>
        </tr>
    """

html_content += """
    </tbody>
</table>
<p>(Note: 'Est. Safety Score' is the midpoint of the Quantile Bin from training. Click the image to view full size.)</p>
"""
display(HTML(html_content))